# Anima — flavor experiments on Colab (thin shell over the repo runner)

Measures **[Anima](https://huggingface.co/circlestone-labs/Anima)** (CircleStone Labs and Comfy Org; 2B, on NVIDIA
Cosmos-Predict2; its text side is Qwen3 0.6B read through a small LLM adapter) as a bed for steering with a second
conditioning source, and uploads each experiment into its own folder of
**[AbstractPhil/geolip-beatrix-anima](https://huggingface.co/AbstractPhil/geolip-beatrix-anima)** as it goes.

**No dataset needed.** The stock model renders everything itself, with the model card's quality prefix and negative
prompt. The mood experiments (32 everyday scenes) are scored by a CLIP mood judge (similarity to upbeat phrases minus
similarity to downbeat phrases), the same judge as the Sana experiments; the attribute screen (eight characters) is
scored by an anime image tagger (WD EVA02-Large v3).

| step | folder | what it asks |
|---|---|---|
| `a.run_flavor_test()` | `e001_anima_flavor_test` | do mood words move the image; does a mood direction added to the conditioning act as a dial, before the LLM adapter and after it; how large are the conditioning vectors there? |
| `a.run_route_split()` | `e020_anima_route_split` | the adapter reads a caption twice, as Qwen3's states and as T5 word ids through its own word table: which of the two carries the mood words? (the mood prompt through one reading at a time; ~25 min) |
| `a.run_appended_split()` | `e021_anima_route_split_appended` | the same with the mood words after the scene: what does the source half add to the query half, above all for the downbeat words? (~25 min) |
| `a.run_query_dial()` | `e022_anima_query_dial` | a mood direction added to the adapter's queries, alone and with one source token a query can look up (a bare direction, or Qwen3's state for the mood words); the direction spread over every Qwen3 token is the control (1,088 images, ~60 min) |
| `a.run_slot_pair()` | `e027_anima_slot_pair` | a word-sized mood push at one word's position, on the adapter's query side, its source side, or both (a matched pair), against the real words at that position (480 images, ~30 min) |
| `a.run_word_split()` | `e026_anima_word_split` | single mood words after the scene, through both readings or the T5 ids only: does the split of e021 follow the mood, or whether the T5 vocabulary holds the word whole? (cheerful / gloomy x one T5 token / cut into pieces; 672 images, ~40 min) |
| `a.run_attribute_screen()` | `e012_anima_attribute_screen` | which character attributes a tag controls (hair length, hair and eye colour, chibi proportions, adult age, style); does each attribute's direction, added after the adapter, act as a slider; does each slider move only its own attribute? |
| `a.run_sequence()` | `e002` ... `e007` | LoRAs trained on the model's own upbeat / neutral / downbeat images captioned neutrally: does it work, the control (can fail), the mirror, a replicate, half and double the learning rate |
| | `e008` ... `e011` (`_draw2`) | the control, the mirror and the two learning rates again on e005's draw of images |
| `a.run_beatrix_connectors()` | `e013` ... `e025` | Beatrix's own states for a mood phrase, turned into a push after the adapter by a small learned map (trained on the LoRAs' images, everything else frozen): does it steer the image, for the phrases it trained on and for phrases it never saw? Against an untrained Beatrix of the same shape (must fail on the unseen phrases) and a free learned vector per mood (the reference). e016 / e017 re-run e013 / e014 with a whitened input (e013's map learned too slowly to read); e018 / e019 with a slider value (where the phrase sits between the gloomy and the cheerful training phrases); e023 / e024 with a two-sided slider (each side of the value its own direction: e018 moved only the cheerful side), e025 with a smooth two-sided one |

**Every long step prints its progress:** the size of the job first, then images or training steps done, time spent
and time left (renders at most every 30 s, training every minute, a line after each finished experiment).

The LoRA recipe follows the model card: Anima-Base v1.0, rank 32 at learning rate 2e-5, the LLM adapter frozen; plain
Adam with no weight decay and fp32 master weights over the bf16 LoRA; 768 px; evaluation at 30 steps, guidance 4.5.
Each arm is scored on 8 scenes it never saw (4 seeds each, 32 paired cells); the rules are fixed in each folder's
README before its run.

**Same thin-shell contract as the other notebooks:** all logic is in the repo
(`geolip_anima_trainer.anima_runner.AnimaRunner`); this notebook is clone -> install -> a few `a.<step>()` calls.
**To change anything, edit the repo and `git pull`; the notebook stays as it is.**

**Before you start:** pick a GPU runtime (the RTX PRO 6000 is the target) and put a **write** `HF_TOKEN` in Colab
Secrets with notebook access: the uploads need it, and every step stops at once if it cannot write.

> **Licence.** Anima's weights are under the CircleStone Labs Non-Commercial License (a derivative of NVIDIA
> Cosmos-Predict2-2B, NVIDIA Open Model License); LoRAs trained on it carry the same non-commercial terms.

## 0 · Confirm the GPU + disk

The run needs about 25 GB of disk at first (the three model files, ~5.6 GB, the dependencies, the training sets) and
grows by a few GB of LoRAs and optimizer states as the arms finish.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!df -h /content | tail -1

## 1 · Bootstrap — clone + install (-> restart ONCE the first time)

Clones this repo and the diffusion-pipe fork (at `external/diffusion-pipe-fork`, which has the previews and the fp32
master weights the recipe uses) and installs the dependencies. **Run the cell; if the kernel restarts, run it again**
(it then skips), then continue.

In [ ]:
import os, sys, subprocess
REPO = os.environ.get("ANIMA_REPO", "/content/anima-trainer")
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "https://github.com/AbstractEyes/anima-trainer.git", REPO], check=True)
else:
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=False)   # the latest runner before importing it
sys.path.insert(0, REPO)
import anima_colab
if anima_colab.install(REPO, dp_url=anima_colab.DP_FORK_URL, similarity=False):   # + the fork
    os.kill(os.getpid(), 9)             # -> restart; reconnect and RE-RUN this cell (then it skips)

## 2 · (after restart) Set up, then run

`setup()` checks the GPU, finds the fork and downloads the three model files. `run_flavor_test()` is e001 (about 700
images, ~40 min). `run_route_split()` is e020 (448 images, ~25 min: the mood words through one of the adapter's two
readings of the caption at a time). `run_attribute_screen()` is e012 (352 images, ~20 min; it downloads the 1.3 GB tagger
and installs `timm` if missing). `run_sequence()` renders the training sets and the no-LoRA baseline first, then trains and evaluates each
arm; every arm uploads its config, data list, each saved epoch, previews, evaluation and log into
`experiments/<folder>/`, and the repo's README index updates after each one. The first arm also checks this notebook's
renderer against the trainer's own preview images (stock and with the LoRA) and records the difference.
`run_beatrix_connectors()` is e013-e025 (about 20 minutes per experiment: a training of 720 steps in this notebook,
then up to 224 images to score; experiments already done are skipped); it reads Beatrix's phrase features from the data
repo, so Beatrix itself never runs here, and it trains on the first-draw training sets of the sequence (downloaded or
drawn if this runtime lacks them).
**A rerun skips every experiment the repo already lists as done**, so after a disconnect just run these cells again.
Run a subset with `a.run_sequence(["e002_lora_mood_up", ...])` or `a.run_beatrix_connectors(["e013_beatrix_mood_connector"])`.

In [ ]:
from geolip_anima_trainer.anima_runner import AnimaRunner
a = AnimaRunner()      # Anima-Base v1.0 at 768 px; uploads to AbstractPhil/geolip-beatrix-anima (repo_id= to change)
a.setup()              # env + HF login + GPU check + the fork + the three model files (~5.6 GB)

In [ ]:
a.run_flavor_test()    # e001: mood words, the dial at two conditioning sites, the conditioning norms

In [ ]:
a.run_route_split()    # e020: the mood words through Qwen3's states only / the T5 ids only (448 images, ~25 min)

In [ ]:
a.run_appended_split() # e021: the same with the mood words after the scene (448 images, ~25 min)
a.run_query_dial()     # e022: a mood direction in the adapter's queries, alone and with a source token (1,088 images)
a.run_word_split()     # e026: single words, one T5 token against cut into pieces (672 images)
a.run_slot_pair()      # e027: a word-sized push at one word's position: query side, source side, both (480 images)

In [ ]:
a.run_attribute_screen()   # e012: attribute words, attribute sliders after the adapter, their cross-talk (~20 min)

In [ ]:
a.run_sequence()       # e002 .. e011 in order; the trainer's log and each arm's reads print here

In [ ]:
a.run_beatrix_connectors()   # e013-e025: a push from Beatrix's phrase features (+ an untrained trunk, a free vector)

## Notes

- **Where things land:** the repo, one folder per experiment (README with the question, the recipe, the rule fixed
  before the run and the result; `meta.json`; `config/`; `data/`; `lora/epochN/`; `samples/`; `eval/`; `logs/`).
  Locally: `/content/anima_data/` (`datasets/`, `experiments/<folder>/`).
- **Training sets are kept** in
  [AbstractPhil/geolip-beatrix-anima-data](https://huggingface.co/datasets/AbstractPhil/geolip-beatrix-anima-data):
  each set uploads once drawn, and a fresh runtime downloads a set drawn with the same settings instead of drawing it
  again (about an hour saved for the six). `a.save_training_sets()` uploads any complete set on this runtime.
- **Beatrix's features** (`beatrix/` in the data repo) were computed once from her checkpoint, so the connector
  experiments need no Beatrix code here. Each connector's weights ship to `experiments/<folder>/connector/` after every
  pass over the training images (W and b, plus the input projection from e016 on; the README of each folder gives the
  push's formula).
- **Using a LoRA:** the files are ComfyUI format: put `experiments/<folder>/lora/epochN/adapter_model.safetensors` in
  `ComfyUI/models/loras/` and load it on Anima-Base.
- **Other Anima versions:** `AnimaRunner(base="preview3")` (the trainer's download choices); the model card recommends
  training on Base.
- **Updating logic = `git pull`** (the bootstrap cell pulls every run); nothing here needs editing.